# Amazon ML Challenge 2026 — cross-encoder on Kaggle GPU

Runs **in parallel** with the main pipeline on your own machine. Two stages:

| Stage | When | What it does | Time on T4 x2 |
|---|---|---|---|
| `train` | **now** | prep → mine hard negatives from train S1s the matcher does NOT use → fine-tune `multilingual-e5-small` (MIT) | ~60–80 min |
| `score` | after your local run finishes **prune** | scores the local candidate pairs (train + test) → `ce_scores.zip` to download | ~60–90 min |

Then locally: unzip `ce_scores.zip` into `work\` and re-run only the `match` stage (the matcher adds the
cross-encoder score + its rank/gap among each S1's candidates as features). Compare the two validation scores
and upload the better `matching_results.tsv`.

**Settings (right panel → Session options):** Accelerator **GPU T4 x2**, Internet **On**, Persistence **Files only**
(so `/kaggle/working` survives between runs). Share both Drive zips as *Anyone with the link*.

**Run the `train` stage with Save Version → Save & Run All** (background, you can close the tab). Its output
(`ce_model/`) then appears in the version's Output tab. For the `score` stage, set `STAGE = "score"`, paste
`EXPORT_ZIP`, add the train version's output as input (*Add Input → Your Work → this notebook*), and run again.

In [ ]:
# ================= EDIT THESE =================
STAGE = "train"            # "train" now; "score" once EXPORT_ZIP (ce_export.zip from your machine) is on Drive
DATA_ZIP = "https://drive.google.com/file/d/1UoYk5KaU-HmVZX2GG1ILejuiD2ULP94I/view?usp=sharing"
CODE_ZIP = ""              # Drive link of the NEW code zip (TEAMNAME_submission.zip)
EXPORT_ZIP = ""            # Drive link of ce_export.zip (score stage only)
MATCHER_TRAIN_FRAC = 0.3   # the --train-frac your local run uses (its S1s are kept out of CE training)
MODEL = "intfloat/multilingual-e5-small"   # MIT, 118M params; alternative: "jhu-clsp/mmBERT-small" (MIT)
N_S1 = 200_000             # train S1s used to mine CE training pairs
TOPK = 8                   # candidates kept per mined S1 (hard negatives)
MAX_PAIRS = 1_500_000
EPOCHS = 1
TRAIN_BATCH, SCORE_BATCH, MAX_LEN = 128, 512, 128
# ===============================================
import os, multiprocessing
N_JOBS = multiprocessing.cpu_count()
ROOT = os.environ.get("ER_ROOT", "/tmp/er")
OUT_ROOT = os.environ.get("ER_OUT", "/kaggle/working")
WORK = f"{ROOT}/work"
MODEL_OUT = f"{OUT_ROOT}/ce_model"
SCORES = f"{OUT_ROOT}/ce_scores"
for d in (ROOT, WORK, SCORES):
    os.makedirs(d, exist_ok=True)
print("stage:", STAGE, "| cores:", N_JOBS)

In [ ]:
!pip install -q gdown sparse_dot_topn rapidfuzz lightgbm anyascii
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not torch.cuda.is_available():
    print("WARNING: no GPU. Session options -> Accelerator -> GPU T4 x2")

In [ ]:
import glob, re, shutil, zipfile

def drive_id(x):
    """Accept a full Drive link or a bare file ID."""
    x = (x or "").strip()
    m = re.search(r"/d/([\w-]{20,})", x) or re.search(r"[?&]id=([\w-]{20,})", x)
    return m.group(1) if m else x

def fetch_from_drive(link, dest):
    if os.path.exists(dest) and zipfile.is_zipfile(dest):
        print("already downloaded:", dest)
        return dest
    import gdown
    fid = drive_id(link)
    print("downloading", fid, "->", dest)
    gdown.download(id=fid, output=dest, quiet=False)
    if not (os.path.exists(dest) and zipfile.is_zipfile(dest)):
        raise RuntimeError(
            "Drive download failed or returned a web page instead of the zip. Check: sharing is 'Anyone with the "
            "link', Internet is On in session options. If Drive says quota exceeded, upload the zip as a Kaggle "
            "Dataset (Add Input -> Upload), clear the link in the config cell and re-run.")
    print(f"ok: {os.path.getsize(dest) / 1e9:.2f} GB")
    return dest

def find_marker(root, marker):
    """Folder that contains `marker` (e.g. 'train/train_source1.tsv'), shallowest first, macOS junk skipped."""
    hits = [h for h in glob.glob(os.path.join(root, "**", marker), recursive=True) if "__MACOSX" not in h]
    if not hits:
        return None
    hit = min(hits, key=len)
    return hit[: -len(marker)].rstrip("/")

def extract_all(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    for inner in glob.glob(os.path.join(dest, "**", "*.zip"), recursive=True):   # zip inside a zip
        if "__MACOSX" not in inner:
            with zipfile.ZipFile(inner) as z:
                z.extractall(os.path.dirname(inner))

def resolve(kind, link, marker):
    target = f"{ROOT}/{kind}"
    found = find_marker(target, marker)
    if found:
        print(f"{kind}: already extracted at {found}")
        return found
    if link.strip():
        zp = fetch_from_drive(link, f"{ROOT}/{kind}.zip")
        extract_all(zp, target)
        os.remove(zp)                      # free disk: the extracted copy is enough
    else:                                  # Kaggle Dataset upload (folder or zip under /kaggle/input)
        found = find_marker("/kaggle/input", marker)
        if found:
            print(f"{kind}: using Kaggle input {found}")
            return found
        zips = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True)]
        for z in zips:
            extract_all(z, target)
    found = find_marker(target, marker)
    if not found:
        raise FileNotFoundError(f"{kind}: could not find {marker} after extraction; check the zip contents")
    print(f"{kind}: {found}")
    return found

DATA_DIR = resolve("data", DATA_ZIP, "train/train_source1.tsv")
if CODE_ZIP.strip():
    shutil.rmtree(f"{ROOT}/code", ignore_errors=True)
CODE_DIR = resolve("code", CODE_ZIP, "src/ce_model.py")
print("DATA_DIR:", DATA_DIR, "\nCODE_DIR:", CODE_DIR)

## Stage `train` (run now)

In [ ]:
def find_model():
    """Model from this session, or from a previous version's output added as input."""
    if os.path.exists(f"{MODEL_OUT}/ce_info.json"):
        return MODEL_OUT
    hits = glob.glob("/kaggle/input/**/ce_model/ce_info.json", recursive=True)
    return os.path.dirname(hits[0]) if hits else None

MODEL_DIR = find_model()
if STAGE == "train" or MODEL_DIR is None:
    if MODEL_DIR is None and STAGE == "score":
        print("no trained model found -> training it first (adds ~60-80 min)")
    if not os.path.exists(f"{WORK}/rec_train.parquet"):
        !cd "{CODE_DIR}" && python -m src.pipeline --data-dir "{DATA_DIR}" --work "{WORK}" --stages prep --set n_jobs={N_JOBS}
    if not os.path.exists(f"{WORK}/ce_mine_pairs.parquet"):
        !cd "{CODE_DIR}" && python -m src.ce_model mine --data-dir "{DATA_DIR}" --work "{WORK}" --matcher-train-frac {MATCHER_TRAIN_FRAC} --n-s1 {N_S1} --topk {TOPK} --n-jobs {N_JOBS}
    !cd "{CODE_DIR}" && python -m src.ce_model train --data-dir "{DATA_DIR}" --pairs "{WORK}/ce_mine_pairs.parquet" --model "{MODEL}" --model-out "{MODEL_OUT}" --max-pairs {MAX_PAIRS} --epochs {EPOCHS} --batch {TRAIN_BATCH} --max-len {MAX_LEN} --workers {N_JOBS}
    MODEL_DIR = find_model()
print("MODEL_DIR:", MODEL_DIR)

## Stage `score` (after your local run has finished `prune` and you uploaded `ce_export.zip`)

In [ ]:
if STAGE == "score":
    import subprocess, time as _t
    EXPORT_DIR = resolve("export", EXPORT_ZIP, "pairs_test.parquet")
    n_gpu = max(torch.cuda.device_count(), 1)          # one independent scoring process per GPU
    for split in ("test", "train"):
        base = ["python", "-m", "src.ce_model", "score", "--data-dir", DATA_DIR, "--split", split,
                "--pairs", f"{EXPORT_DIR}/pairs_{split}.parquet", "--align", f"{EXPORT_DIR}/align.json",
                "--model-dir", MODEL_DIR, "--out", f"{SCORES}/ce_{split}.parquet",
                "--shard-dir", f"{OUT_ROOT}/ce_shards_{split}", "--batch", str(SCORE_BATCH),
                "--workers", str(max(1, N_JOBS // n_gpu))]
        procs, logs = [], []
        for i in range(n_gpu):
            env = dict(os.environ)
            if torch.cuda.is_available():
                env["CUDA_VISIBLE_DEVICES"] = str(i)
            logs.append(f"{ROOT}/score_{split}_{i}.log")
            procs.append(subprocess.Popen(base + ["--part", str(i), "--parts", str(n_gpu)], cwd=CODE_DIR, env=env,
                                          stdout=open(logs[-1], "w"), stderr=subprocess.STDOUT))
        while any(p.poll() is None for p in procs):
            _t.sleep(60)
            for i, lg in enumerate(logs):
                lines = open(lg).read().strip().splitlines()
                print(f"[{split} gpu{i}]", lines[-1][:170] if lines else "starting")
        for i, p in enumerate(procs):
            if p.returncode != 0:
                print(open(logs[i]).read()[-3000:])
                raise RuntimeError(f"scoring {split} part {i} failed (log above)")
        subprocess.run(base + ["--parts", "1"], cwd=CODE_DIR, check=True)   # merge the shards
    zp = shutil.make_archive(f"{OUT_ROOT}/ce_scores", "zip", SCORES, ".")
    for f in ("ce_train.parquet", "ce_test.parquet"):
        print(f, f"{os.path.getsize(f'{SCORES}/{f}') / 1e6:.0f} MB")
    print("download:", zp, "-> unzip into your local work folder")
else:
    print("train stage finished; model at", MODEL_DIR)

In [ ]:
# free space in the saved output: shards are only needed to resume an interrupted scoring run
if STAGE == "score" and os.path.exists(f"{OUT_ROOT}/ce_scores.zip"):
    for s in glob.glob(f"{OUT_ROOT}/ce_shards_*"):
        shutil.rmtree(s, ignore_errors=True)
    shutil.rmtree(SCORES, ignore_errors=True)
print(sorted(os.listdir(OUT_ROOT)))